# Homework: worked answers

Five questions, each answered both ways, with the two compared.

In [ ]:
import os
import sqlite3
from pathlib import Path

import pandas as pd

here = Path.cwd()
while not (here / "data" / "music.db").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

con = sqlite3.connect("data/music.db")
plays = pd.read_sql("SELECT * FROM plays", con, parse_dates=["played_at"])
artists = pd.read_sql("SELECT * FROM artists", con)

pd.set_option("display.width", 110)

## 1. Plays on a laptop in 2024

Answer: 263.

In [ ]:
sql = pd.read_sql("""
    SELECT COUNT(*) AS n
    FROM plays
    WHERE device = 'laptop'
      AND played_at BETWEEN '2024-01-01' AND '2024-12-31'
""", con)["n"][0]

pandas_way = len(plays[(plays["device"] == "laptop")
                       & (plays["played_at"].dt.year == 2024)])

print(sql, pandas_way, "| agree:", sql == pandas_way)

`.dt.year` is nicer than the SQL date range, and it is only available because
`parse_dates` turned the column into real dates when we loaded it. On a plain
text column you would be back to string comparisons.

## 2. Three countries with the most total minutes

Germany, the Netherlands, Norway.

In [ ]:
sql = pd.read_sql("""
    SELECT a.country, ROUND(SUM(p.minutes_played), 1) AS minutes
    FROM plays p
    JOIN artists a USING (artist_id)
    GROUP BY a.country
    ORDER BY minutes DESC
    LIMIT 3
""", con)

pandas_way = (plays.merge(artists, on="artist_id")
                   .groupby("country")["minutes_played"]
                   .sum()
                   .round(1)
                   .nlargest(3)
                   .reset_index(name="minutes"))

print(sql.to_string(index=False))
print()
print(pandas_way.to_string(index=False))

## 3. Plays and average length per genre

In [ ]:
sql = pd.read_sql("""
    SELECT a.genre,
           COUNT(*)                        AS plays,
           ROUND(AVG(p.minutes_played), 2) AS avg_min
    FROM plays p
    JOIN artists a USING (artist_id)
    GROUP BY a.genre
    ORDER BY plays DESC
""", con)

pandas_way = (plays.merge(artists, on="artist_id")
                   .groupby("genre")
                   .agg(plays=("play_id", "count"),
                        avg_min=("minutes_played", "mean"))
                   .round(2)
                   .sort_values("plays", ascending=False)
                   .reset_index())

print(sql.to_string(index=False))
print()
print(pandas_way.to_string(index=False))

## 4. Artists never played

Iron Fernway and Gravel Choir.

This one is about the join type, in both languages. An inner join cannot
answer it, because the answer is made entirely of rows with no match.

In [ ]:
sql = pd.read_sql("""
    SELECT a.artist_name, a.country
    FROM artists a
    LEFT JOIN plays p USING (artist_id)
    WHERE p.play_id IS NULL
""", con)

merged = artists.merge(plays, on="artist_id", how="left")
pandas_way = merged[merged["play_id"].isna()][["artist_name", "country"]]

print(sql.to_string(index=False))
print()
print(pandas_way.to_string(index=False))

Note the pattern in both: **left join, then keep only the rows where the
other side is empty.** `WHERE ... IS NULL` in SQL, `.isna()` in pandas. It is
the standard way to ask "which of these has none of those".

## 5. Percentage of plays skipped, per device

In [ ]:
sql = pd.read_sql("""
    SELECT device,
           COUNT(*)     AS plays,
           SUM(skipped) AS skips,
           ROUND(100.0 * SUM(skipped) / COUNT(*), 1) AS skipped_pct
    FROM plays
    GROUP BY device
    ORDER BY skipped_pct DESC
""", con)

pandas_way = (plays.groupby("device")
                   .agg(plays=("play_id", "count"),
                        skips=("skipped", "sum"),
                        skipped_pct=("skipped", "mean"))
                   .assign(skipped_pct=lambda d: (d["skipped_pct"] * 100).round(1))
                   .sort_values("skipped_pct", ascending=False)
                   .reset_index())

print(sql.to_string(index=False))
print()
print(pandas_way.to_string(index=False))

Two things worth noticing.

**In SQL, `100.0` matters.** `SUM(skipped) / COUNT(*)` with two whole numbers
is 0, because SQLite throws the fraction away.

**In pandas, `.mean()` on a column of 0s and 1s is the proportion directly.**
No division needed. That is a genuinely handy trick: a yes/no column stored
as 0 and 1 gives you its rate for free.

---

# Task 2: the awkward one

**For each genre, what share of its total minutes came from its single most
played artist?**

In [ ]:
both = plays.merge(artists, on="artist_id")

# Step 1: minutes per artist, within each genre.
per_pair = (both.groupby(["genre", "artist_name"])["minutes_played"]
                .sum()
                .reset_index())

# Step 2: for each genre, the total, and the biggest single artist.
summary = (per_pair.groupby("genre")["minutes_played"]
                   .agg(total="sum", top="max")
                   .reset_index())

# Step 3: the share, and who it was.
summary["share_pct"] = (100 * summary["top"] / summary["total"]).round(1)

top_artist = (per_pair.sort_values("minutes_played", ascending=False)
                      .drop_duplicates("genre")[["genre", "artist_name"]])

answer = (summary.merge(top_artist, on="genre")
                 .sort_values("share_pct", ascending=False)
                 .round(1))

print(answer.to_string(index=False))

## Why that was awkward in SQL

The pandas version is three groupbys and a division. Nothing clever.

In SQL you need a per-genre **total** and a per-genre **maximum** at the same
time, which means either two CTEs joined together, or a window function:

```sql
WITH per_pair AS (
    SELECT a.genre, a.artist_name, SUM(p.minutes_played) AS minutes
    FROM plays p JOIN artists a USING (artist_id)
    GROUP BY a.genre, a.artist_name
)
SELECT genre,
       MAX(minutes) AS top,
       SUM(minutes) AS total,
       ROUND(100.0 * MAX(minutes) / SUM(minutes), 1) AS share_pct
FROM per_pair
GROUP BY genre
ORDER BY share_pct DESC;
```

That is not bad at all, as it turns out. Getting the artist's **name**
alongside it is where it gets genuinely fiddly, because `MAX(minutes)` tells
you the number but not which row it came from. In pandas that is
`sort_values().drop_duplicates()`, two methods.

Let us check the SQL version agrees on the numbers:

In [ ]:
sql_version = pd.read_sql("""
    WITH per_pair AS (
        SELECT a.genre, a.artist_name, SUM(p.minutes_played) AS minutes
        FROM plays p JOIN artists a USING (artist_id)
        GROUP BY a.genre, a.artist_name
    )
    SELECT genre,
           ROUND(SUM(minutes), 1) AS total,
           ROUND(MAX(minutes), 1) AS top,
           ROUND(100.0 * MAX(minutes) / SUM(minutes), 1) AS share_pct
    FROM per_pair
    GROUP BY genre
    ORDER BY share_pct DESC
""", con)

print(sql_version.to_string(index=False))

The honest conclusion: **neither tool is better, and the awkwardness moves
around.** Aggregating is easy in both. Getting the row that produced a
maximum is easy in pandas and needs a window function in SQL. Filtering
millions of rows before you ever touch Python is easy in SQL and impossible
in pandas.

Which is why `pd.read_sql` is the answer more often than either one alone.

In [ ]:
con.close()